# Task 3: Local Serving Optimization & Concurrency Capacity Planning
**Applied Generative AI Engineering & System Optimization**
*Target Model: Qwen2.5 Family | Hardware: NVIDIA CUDA GPU*

This notebook executes:
1. Local hardware benchmarking of FP16 vs 4-bit NormalFloat (NF4) quantization.
2. VRAM footprint profiling (Weights VRAM, Peak Generation VRAM).
3. Generation throughput profiling (Decoding Tokens/sec, Latency).
4. Side-by-side qualitative output analysis across 10 technical prompts.
5. In-depth conceptual explanations of the KV Cache and Quantization mechanics.
6. Mathematically rigorous 20-concurrent-user serving plan on a single 24 GB GPU.


In [1]:
# 1. Environment Setup & Hardware Inspection
!pip install -q torch transformers bitsandbytes accelerate

import torch
import json

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**2):.1f} MB")


CUDA Available: True
Device: NVIDIA GeForce RTX 4050 Laptop GPU
Total VRAM: 6141.0 MB


## 2. Benchmark FP16 vs 4-bit NF4 Quantization
Evaluates 10 identical technical prompts spanning metadata extraction, code generation, mathematical derivations, and conceptual systems architecture.


In [2]:
# Run inference benchmark
!python ../task3_optimization/benchmark_inference.py


TASK 3 INFERENCE BENCHMARK RESULTS (NVIDIA GeForce RTX 4050 Laptop GPU):
Setting A: FP16 (16-bit Float Baseline)
  Model Weights Memory:    942.32 MB
  Peak Generation Memory:  957.62 MB
  Average Decoding Speed:  7.97 tokens/sec
  Total Generation Time:   168.93 seconds (10 prompts)

Setting B: 4-bit NormalFloat (NF4 BitsAndBytes Quantized)
  Model Weights Memory:    436.35 MB  (2.16x compression, -53.7%)
  Peak Generation Memory:  457.66 MB  (2.09x compression, -52.2%)
  Average Decoding Speed:  5.19 tokens/sec  (-34.9% decode throughput)
  Total Generation Time:   255.47 seconds (10 prompts)
Qualitative Output Comparison:
  All 10 prompts produced structurally valid, mathematically accurate answers.
  NF4 showed 0 degradation in reasoning, coding, or JSON structure.
Results saved to task3_optimization/inference_benchmark_results.json


## 3. Core Conceptual Explanations in Own Words
- **What the KV Cache Is**: In autoregressive generation, producing token $t$ requires attention over all previous tokens $1 \dots t-1$. Recomputing key ($K$) and value ($V$) projection matrices at every step causes an $O(T^2)$ computational disaster. The KV cache stores previously computed $K$ and $V$ tensors in GPU memory across all layers. At step $t$, the GPU only computes $Q, K, V$ for the single newest token, retrieves past $K, V$ states from cache, and executes matrix-vector attention in $O(T)$ time. It trades memory capacity for massive latency reduction.
- **Why Quantization Saves Memory**: Neural networks are overparameterized; their weights natively reside in 32-bit (FP32) or 16-bit (FP16/BF16) floats requiring 4 or 2 bytes each. Quantization projects these continuous values onto discrete, low-bit representations (e.g. 4-bit integers or NormalFloat4) via scale ($S$) and zero-point ($Z$): $W_q = \text{round}(W / S) + Z$. Storing weights in 4 bits reduces physical memory consumption by **75% vs FP16** and **87.5% vs FP32**, while shrinking memory-bandwidth pressure across the GPU memory bus.


## 4. Serving 20 Concurrent Users on a 24 GB GPU (Mathematical Plan)
*Target: Serve 20 concurrent users on a single 24 GB GPU (NVIDIA RTX 3090 / A10G, 24,576 MB VRAM).*

### 1. Workload & Model Parameters
- **Model**: `Qwen2.5-7B-Instruct` quantized to 4-bit AWQ / GPTQ ($N = 7.0 \times 10^9$ parameters).
- **Architecture**: Number of layers $L = 28$, hidden dimension $d = 3,584$, Grouped-Query Attention (GQA) with $H_Q = 28$ query heads and $H_{KV} = 4$ key-value heads, head dimension $d_k = 128$.
- **User Workload**: 20 active concurrent streams. Context budget per user = **2,048 tokens** (1,536 input prompt + 512 generation).
- **KV Cache Precision**: 16-bit FP16.

### 2. Physical Memory Budget Breakdown
$$\text{Available VRAM} = 24.00\text{ GB} = 24,576\text{ MB}$$
1. **Static Model Weights (4-bit AWQ)**: $7.0 \times 10^9 \times 0.5\text{ bytes} = 3.50\text{ GB} + 0.35\text{ GB (scaling scales/zeros)} = \mathbf{3.85\text{ GB}}$ ($3,942\text{ MB}$).
2. **CUDA Context & Runtime Scratchpad**: PyTorch context, kernels, and BLAS scratchpads = $\mathbf{1.20\text{ GB}}$ ($1,228\text{ MB}$).
3. **Peak Dynamic Activation Working Buffer**: Peak intermediate activations during chunked prefill ($B=1, S=512$) = $\mathbf{1.95\text{ GB}}$ ($2,000\text{ MB}$).
4. **Dedicated Dynamic KV Cache Pool**:
   $$\text{VRAM}_{\text{KV}} = 24.00\text{ GB} - 3.85\text{ GB} - 1.20\text{ GB} - 1.95\text{ GB} = \mathbf{17.00\text{ GB}} \quad (17,408\text{ MB} = 18,253,611,008\text{ bytes})$$

### 3. KV Cache Capacity & Headroom Derivation
Under Grouped-Query Attention ($H_{KV} = 4$), the memory required to store the KV cache for **one token** across all 28 layers is:
$$\text{Bytes per Token} = 2 \times L \times H_{KV} \times d_k \times 2\text{ bytes (FP16)} = 2 \times 28 \times 4 \times 128 \times 2 = \mathbf{57,344\text{ bytes}} \approx \mathbf{0.0546875\text{ MB}}$$
- **Total System Token Capacity**: $\frac{18,253,611,008\text{ bytes}}{57,344\text{ bytes/token}} = \mathbf{318,318\text{ tokens}}$.
- **Peak Concurrency Requirement (20 Users)**: $20\text{ users} \times 2,048\text{ tokens} = \mathbf{40,960\text{ tokens}}$.
- **Total VRAM Consumed by 20 Users**: $40,960 \times 57,344\text{ bytes} \approx \mathbf{2.24\text{ GB}}$ ($2,348\text{ MB}$).
- **Safety Headroom Factor**:
  $$\text{Safety Ratio} = \frac{318,318\text{ max tokens}}{40,960\text{ required tokens}} = \mathbf{7.77\times \text{ Headroom Safety Buffer}}$$

### 4. Production Serving Stack Implementation
- **PagedAttention (vLLM Engine)**: Eliminates virtual memory fragmentation by allocating KV cache in discrete 16-token physical blocks, boosting memory utilization from $60\%$ to $96\%$.
- **Continuous (Iteration-Level) Batching**: Requests join and exit the running batch dynamically at each decode step, keeping compute utilization at saturation without waiting for the slowest user to finish.
- **Chunked Prefill & CPU Offload Swap**: Long prompts (1,536 tokens) are chunked into 512-token segments, preventing high-latency prefill spikes from stalling active generation streams. If a user exceeds budget, inactive blocks are swapped to host system RAM over PCIe.
